# Distributed Computing with Ray on GKE from Kubeflow Notebooks v2

This notebook demonstrates how to use a **Kubeflow Notebook v2** (`Workspace`) as an interactive control plane to provision, connect to, submit distributed workloads to, and tear down a **KubeRay `RayCluster`** on Google Kubernetes Engine (GKE).

## Architecture

```
  ┌─────────────────────────────────────────────────────────────────┐
  │       Kubeflow Notebook v2 Pod (Tiny CPU: 0.1 CPU, 128Mi)       │
  │         ServiceAccount: ws-<workspace-name> (ray-edit)          │
  └───────────────┬─────────────────────────────────┬───────────────┘
                  │ 1. Create / Delete RayCluster   │ 2. Submit Tasks / Jobs
                  ▼ (Kubernetes CustomObjectsApi)   ▼ (Ray Client :10001 / Jobs API :8265)
  ┌───────────────────────────────┐ ┌───────────────────────────────────────────────┐
  │    Kubernetes API Server      │ │           RayCluster (ray.io/v1)              │
  │      + KubeRay Operator       │ │  ┌─────────────────┐   ┌───────────────────┐  │
  └───────────────────────────────┘ │  │  Ray Head Pod   │   │  Ray Worker Pods  │  │
                                    │  │ (GCS / Dash /   │──▶│  (Distributed     │  │
                                    │  │  Client Server) │   │   Task Execution) │  │
                                    │  └─────────────────┘   └───────────────────┘  │
                                    └───────────────────────────────────────────────┘
```

### Why this pattern?
1. **Cost Efficiency**: Your JupyterLab Workspace runs on a minimal CPU pod (`tiny_cpu`) all day while you write code and analyze results. Heavy compute nodes (`RayCluster` worker pods with CPUs/GPUs/TPUs) are spun up on demand and deleted as soon as your job finishes.
2. **Zero Credential Plumbing**: Kubeflow Notebooks v2 automatically binds the `ray-edit` `ClusterRole` to the Workspace's in-cluster `ServiceAccount` (`ws-<workspace-name>`), allowing your notebook to manage `RayCluster` resources strictly within your namespace.

## Step 0: Environment & Package Check

First, we ensure the `ray[default,client]` and `kubernetes` Python packages are installed, and verify our in-cluster Kubernetes identity and namespace.

In [ ]:
import os
import sys
import subprocess

# Ensure matching Ray version (2.41.0) and Kubernetes Python client are installed
RAY_VERSION = "2.41.0"
try:
    import ray
    import kubernetes
    assert ray.__version__ == RAY_VERSION
except (ImportError, AssertionError):
    print(f"Installing ray[default,client]=={RAY_VERSION} and kubernetes...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        f"ray[default,client]=={RAY_VERSION}",
        "kubernetes>=28.1.0",
    ])
    import ray
    import kubernetes

from kubernetes import client, config

# Load in-cluster ServiceAccount credentials mounted by Kubernetes
config.load_incluster_config()

# Detect current Workspace namespace
SA_NAMESPACE_FILE = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(SA_NAMESPACE_FILE):
    with open(SA_NAMESPACE_FILE, "r") as f:
        NAMESPACE = f.read().strip()
else:
    NAMESPACE = "default"

print(f"Python version : {sys.version.split()[0]}")
print(f"Ray version    : {ray.__version__}")
print(f"Namespace      : {NAMESPACE}")

## Step 1: Define a Helper Function to Create & Manage a `RayCluster`

Using the permissions granted by the `ray-edit` `ClusterRole` (configured in `WorkspaceKind.spec.podTemplate.serviceAccount.clusterRoles`), our notebook can directly create, update, and delete `RayCluster` (`ray.io/v1`) custom resources in its namespace.

Below we define `create_ray_cluster()` and `delete_ray_cluster()` with customizable options for:
- `cluster_name`, `ray_version`, `image` (defaults to matching the notebook's Python version)
- `num_workers`, `min_workers`, `max_workers`
- Head & worker CPU / memory requests and limits
- Optional GPU acceleration (`worker_gpu`)
- Automatic Istio sidecar exclusion (`sidecar.istio.io/inject: "false"`)
- Readiness polling returning `(RAY_CLIENT_URI, RAY_DASHBOARD_URI)`

In [ ]:
import time
from typing import Optional

def create_ray_cluster(
    cluster_name: str = "raycluster-sample",
    namespace: str = NAMESPACE,
    ray_version: str = RAY_VERSION,
    num_workers: int = 2,
    min_workers: Optional[int] = None,
    max_workers: Optional[int] = None,
    head_cpu_request: str = "500m",
    head_cpu_limit: str = "1000m",
    head_memory_request: str = "2Gi",
    head_memory_limit: str = "4Gi",
    worker_cpu_request: str = "1000m",
    worker_cpu_limit: str = "2000m",
    worker_memory_request: str = "2Gi",
    worker_memory_limit: str = "4Gi",
    worker_gpu: int = 0,
    image: Optional[str] = None,
    wait_for_ready: bool = True,
    timeout_seconds: int = 300,
) -> tuple[str, str]:
    """Create or update a KubeRay RayCluster in Kubernetes and optionally wait until ready.

    Returns:
        (ray_client_uri, ray_dashboard_uri)
    """
    py_tag = f"py{sys.version_info.major}{sys.version_info.minor}"
    if image is None:
        suffix = "-gpu" if worker_gpu > 0 else ""
        image = f"rayproject/ray:{ray_version}-{py_tag}{suffix}"

    min_replicas = min_workers if min_workers is not None else num_workers
    max_replicas = max_workers if max_workers is not None else max(num_workers, min_replicas)

    worker_resources = {
        "requests": {"cpu": worker_cpu_request, "memory": worker_memory_request},
        "limits": {"cpu": worker_cpu_limit, "memory": worker_memory_limit},
    }
    worker_tolerations = []
    if worker_gpu > 0:
        worker_resources["requests"]["nvidia.com/gpu"] = str(worker_gpu)
        worker_resources["limits"]["nvidia.com/gpu"] = str(worker_gpu)
        worker_tolerations.append({
            "key": "nvidia.com/gpu",
            "operator": "Exists",
            "effect": "NoSchedule",
        })

    manifest = {
        "apiVersion": "ray.io/v1",
        "kind": "RayCluster",
        "metadata": {
            "name": cluster_name,
            "namespace": namespace,
            "labels": {
                "app.kubernetes.io/name": cluster_name,
                "app.kubernetes.io/part-of": "kubeflow-notebooks-ray",
            },
        },
        "spec": {
            "rayVersion": ray_version,
            "headGroupSpec": {
                "serviceType": "ClusterIP",
                "rayStartParams": {
                    "dashboard-host": "0.0.0.0",
                    "num-cpus": "0",  # Keep head node dedicated to cluster coordination
                },
                "template": {
                    "metadata": {
                        "labels": {
                            "ray.io/node-type": "head",
                            # Disable Istio sidecar injection so Ray internal gRPC traffic on
                            # dynamic ephemeral ports is not blocked by Istio AuthorizationPolicy
                            "sidecar.istio.io/inject": "false",
                        }
                    },
                    "spec": {
                        "containers": [
                            {
                                "name": "ray-head",
                                "image": image,
                                "ports": [
                                    {"containerPort": 6379, "name": "gcs-server"},
                                    {"containerPort": 8265, "name": "dashboard"},
                                    {"containerPort": 10001, "name": "client"},
                                ],
                                "resources": {
                                    "requests": {"cpu": head_cpu_request, "memory": head_memory_request},
                                    "limits": {"cpu": head_cpu_limit, "memory": head_memory_limit},
                                },
                            }
                        ]
                    },
                },
            },
            "workerGroupSpecs": [
                {
                    "groupName": "worker-group",
                    "replicas": num_workers,
                    "minReplicas": min_replicas,
                    "maxReplicas": max_replicas,
                    "rayStartParams": {},
                    "template": {
                        "metadata": {
                            "labels": {
                                "ray.io/node-type": "worker",
                                "sidecar.istio.io/inject": "false",
                            }
                        },
                        "spec": {
                            "tolerations": worker_tolerations,
                            "containers": [
                                {
                                    "name": "ray-worker",
                                    "image": image,
                                    "resources": worker_resources,
                                }
                            ],
                        },
                    },
                }
            ],
        },
    }

    custom_api = client.CustomObjectsApi()

    try:
        custom_api.create_namespaced_custom_object(
            group="ray.io",
            version="v1",
            namespace=namespace,
            plural="rayclusters",
            body=manifest,
        )
        print(f"Created RayCluster '{cluster_name}' in namespace '{namespace}' (workers={num_workers}, image={image}).")
    except client.exceptions.ApiException as e:
        if e.status == 409:
            print(f"RayCluster '{cluster_name}' already exists in namespace '{namespace}', patching spec...")
            custom_api.patch_namespaced_custom_object(
                group="ray.io",
                version="v1",
                namespace=namespace,
                plural="rayclusters",
                name=cluster_name,
                body={"spec": manifest["spec"]},
            )
        else:
            raise

    head_svc = f"{cluster_name}-head-svc.{namespace}.svc.cluster.local"
    ray_client_uri = f"ray://{head_svc}:10001"
    ray_dashboard_uri = f"http://{head_svc}:8265"

    if wait_for_ready:
        print(f"Waiting for RayCluster '{cluster_name}' to become ready...")
        start_time = time.time()
        while time.time() - start_time < timeout_seconds:
            cr = custom_api.get_namespaced_custom_object(
                group="ray.io",
                version="v1",
                namespace=namespace,
                plural="rayclusters",
                name=cluster_name,
            )
            status = cr.get("status", {})
            state = status.get("state", "pending")
            available_workers = status.get("availableWorkerReplicas", 0)
            elapsed = int(time.time() - start_time)
            print(f"  [{elapsed:3d}s] State: {state:<10} | Available Workers: {available_workers}/{num_workers}")
            if state == "ready" and available_workers >= min(1, num_workers):
                print("\n✅ RayCluster is READY!")
                print(f"   Ray Client URI    : {ray_client_uri}")
                print(f"   Ray Dashboard URI : {ray_dashboard_uri}")
                return ray_client_uri, ray_dashboard_uri
            time.sleep(5)
        raise TimeoutError(f"Timed out waiting for RayCluster '{cluster_name}' after {timeout_seconds}s.")

    return ray_client_uri, ray_dashboard_uri


def delete_ray_cluster(
    cluster_name: str = "raycluster-sample",
    namespace: str = NAMESPACE,
) -> None:
    """Delete a KubeRay RayCluster custom resource from Kubernetes."""
    custom_api = client.CustomObjectsApi()
    try:
        custom_api.delete_namespaced_custom_object(
            group="ray.io",
            version="v1",
            namespace=namespace,
            plural="rayclusters",
            name=cluster_name,
        )
        print(f"🗑️ Deleted RayCluster '{cluster_name}' in namespace '{namespace}'.")
    except client.exceptions.ApiException as e:
        if e.status == 404:
            print(f"RayCluster '{cluster_name}' already deleted.")
        else:
            raise

## Step 2: Provision the `RayCluster` & Wait for Readiness

Now we call `create_ray_cluster()` to spin up a 2-worker Ray cluster and wait for the KubeRay operator to expose the Head Pod endpoints:
- **Port `10001`**: Ray Client server (`ray://`)
- **Port `8265`**: Ray Dashboard & Jobs API server (`http://`)
- **Port `6379`**: Ray GCS server

In [ ]:
import time

HEAD_SVC = f"{CLUSTER_NAME}-head-svc.{NAMESPACE}.svc.cluster.local"
RAY_CLIENT_URI = f"ray://{HEAD_SVC}:10001"
RAY_DASHBOARD_URI = f"http://{HEAD_SVC}:8265"

print(f"Waiting for RayCluster '{CLUSTER_NAME}' to become ready...")
for attempt in range(60):
    cr = custom_api.get_namespaced_custom_object(
        group="ray.io",
        version="v1",
        namespace=NAMESPACE,
        plural="rayclusters",
        name=CLUSTER_NAME,
    )
    status = cr.get("status", {})
    state = status.get("state", "pending")
    available_workers = status.get("availableWorkerReplicas", 0)
    print(f"  [{attempt * 5:3d}s] State: {state:<10} | Available Workers: {available_workers}")
    if state == "ready" and available_workers >= 1:
        print("\n✅ RayCluster is READY!")
        print(f"   Ray Client URI    : {RAY_CLIENT_URI}")
        print(f"   Ray Dashboard URI : {RAY_DASHBOARD_URI}")
        break
    time.sleep(5)
else:
    raise TimeoutError(f"Timed out waiting for RayCluster '{CLUSTER_NAME}' to become ready.")

## Step 3: Interactive Development — Connect via Ray Client & Run Distributed Tasks

We connect our notebook session directly to the `RayCluster` using `ray.init("ray://<head-svc>:10001")`. Once connected, any function or class decorated with `@ray.remote` is automatically serialized and executed across the worker pods in the cluster.

In [ ]:
import socket
import random
import ray

# Connect to the RayCluster via Ray Client protocol
if ray.is_initialized():
    ray.shutdown()

ray.init(address=RAY_CLIENT_URI)

print("Connected to RayCluster!")
print("Cluster Resources:", ray.cluster_resources())

# 1. Define a stateless @ray.remote task
@ray.remote
def estimate_pi_chunk(num_samples: int) -> tuple[int, str]:
    """Monte Carlo pi estimation running on a remote Ray worker pod."""
    inside = 0
    for _ in range(num_samples):
        x, y = random.random(), random.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return inside, socket.gethostname()

# Submit 8 parallel tasks across the worker pods
NUM_TASKS = 8
SAMPLES_PER_TASK = 2_000_000

futures = [estimate_pi_chunk.remote(SAMPLES_PER_TASK) for _ in range(NUM_TASKS)]
results = ray.get(futures)

total_inside = sum(count for count, _ in results)
worker_hosts = {host for _, host in results}
pi_estimate = (4.0 * total_inside) / (NUM_TASKS * SAMPLES_PER_TASK)

print(f"\nEstimated Pi : {pi_estimate:.6f}")
print(f"Executed across worker pods: {sorted(worker_hosts)}")

# 2. Define a stateful @ray.remote Actor
@ray.remote
class ParameterServer:
    def __init__(self, dim: int):
        self.weights = [0.0] * dim

    def push_gradients(self, grads: list[float]) -> list[float]:
        self.weights = [w + g for w, g in zip(self.weights, grads)]
        return self.weights

    def get_weights(self) -> list[float]:
        return self.weights

ps = ParameterServer.remote(dim=4)
update_futures = [ps.push_gradients.remote([0.1 * i, 0.2 * i, 0.3 * i, 0.4 * i]) for i in range(1, 5)]
ray.get(update_futures)
print("Updated ParameterServer weights:", ray.get(ps.get_weights.remote()))

# Disconnect interactive client session
ray.shutdown()

## Step 4: Submit Asynchronous Batch Jobs via Ray Jobs API

For production or long-running training runs, use the **Ray Jobs API** (`JobSubmissionClient`). Unlike interactive Ray Client connections, jobs submitted via `JobSubmissionClient` run completely independently on the cluster—even if you close your browser or pause your notebook Workspace!

In [ ]:
from ray.job_submission import JobSubmissionClient, JobStatus

# Create a local working directory with an entrypoint script to submit
os.makedirs("ray_job_src", exist_ok=True)
with open("ray_job_src/train_job.py", "w") as f:
    f.write('''import ray
import socket
import time

ray.init()

@ray.remote
def distributed_step(epoch: int, shard_id: int):
    time.sleep(0.5)
    loss = 1.0 / (epoch + shard_id + 1)
    return {"epoch": epoch, "shard": shard_id, "loss": round(loss, 4), "pod": socket.gethostname()}

print("Starting distributed training job on Ray cluster...")
for epoch in range(1, 4):
    step_results = ray.get([distributed_step.remote(epoch, s) for s in range(4)])
    avg_loss = sum(r["loss"] for r in step_results) / len(step_results)
    pods = sorted({r["pod"] for r in step_results})
    print(f"Epoch {epoch} | Avg Loss: {avg_loss:.4f} | Worker Pods: {pods}")

print("Training job completed successfully!")
''')

job_client = JobSubmissionClient(RAY_DASHBOARD_URI)

job_id = job_client.submit_job(
    entrypoint="python train_job.py",
    runtime_env={"working_dir": "./ray_job_src"},
)
print(f"Submitted Ray Job ID: {job_id}")

# Poll until job finishes
while True:
    status = job_client.get_job_status(job_id)
    print(f"  Job status: {status}")
    if status in {JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.STOPPED}:
        break
    time.sleep(2)

print("\n--- Job Logs ---")
print(job_client.get_job_logs(job_id))

## Step 5: Tear Down / Delete `RayCluster`

When your distributed computation is complete, delete the `RayCluster` custom resource to release the worker and head pods back to the GKE cluster while keeping your lightweight notebook running.

In [ ]:
try:
    custom_api.delete_namespaced_custom_object(
        group="ray.io",
        version="v1",
        namespace=NAMESPACE,
        plural="rayclusters",
        name=CLUSTER_NAME,
    )
    print(f"🗑️ Deleted RayCluster '{CLUSTER_NAME}' in namespace '{NAMESPACE}'.")
except client.exceptions.ApiException as e:
    if e.status == 404:
        print(f"RayCluster '{CLUSTER_NAME}' already deleted.")
    else:
        raise